# CLIP Reproduction — Evaluation & Visualisation

This notebook lets you:
1. **Query the model** — type any text, see the most relevant Flickr30k images
2. **Inspect the similarity matrix** — see how positive pairs (diagonal) stand out from negatives
3. **Measure retrieval quality** — R@1 and R@5 across the full validation set

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from pathlib import Path
from transformers import T5Tokenizer
from datasets import load_dataset
from tqdm.notebook import tqdm
import random

from model import CLIP
from data import VAL_TRANSFORM

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.family': 'sans-serif'})

Skipping import of cpp extensions due to incompatible torch version 2.11.0 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0529 18:01:43.676000 12483 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


## 1. Load model + dataset

In [2]:
CHECKPOINT = "clip_final.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
model = CLIP(embed_dim=256).to(DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded checkpoint  |  temperature τ = {model.temperature.item():.3f}")

tokenizer = T5Tokenizer.from_pretrained("t5-small")

print("Loading Flickr30k...")
ds = load_dataset("nlphuji/flickr30k", trust_remote_code=True)["test"]
print(f"Dataset: {len(ds)} images")

Device: cpu


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/ethanhersch/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:03<00:00, 14.1MB/s]


...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loading Flickr30k...


OSError: Not enough disk space. Needed: 8.13 GiB (download: 4.10 GiB, generated: 4.03 GiB, post-processed: Unknown size)

## 2. Build image index (runs once, ~2 min)

In [ ]:
INDEX_PATH = Path("image_index.pt")

def build_index(batch_size=128):
    all_emb = []
    all_captions = []
    for start in tqdm(range(0, len(ds), batch_size), desc="Encoding images"):
        rows = ds.select(range(start, min(start + batch_size, len(ds))))
        imgs = []
        for row in rows:
            img = row["image"].convert("RGB")
            imgs.append(VAL_TRANSFORM(img))
            all_captions.append(row["caption"][0])
        with torch.no_grad():
            emb = model.encode_image(torch.stack(imgs).to(DEVICE))
        all_emb.append(emb.cpu())
    emb_tensor = torch.cat(all_emb)
    torch.save({"embeddings": emb_tensor, "captions": all_captions}, INDEX_PATH)
    return emb_tensor, all_captions

if INDEX_PATH.exists():
    print("Loading existing index...")
    data = torch.load(INDEX_PATH)
    img_emb, captions = data["embeddings"], data["captions"]
else:
    img_emb, captions = build_index()

print(f"Index: {img_emb.shape}  ({img_emb.shape[0]} images × {img_emb.shape[1]}-dim)")

---
## 3. Text → Image retrieval

Run `query(...)` with any text. Green border = top match.

In [ ]:
def encode_text(text):
    tokens = tokenizer(text, max_length=64, padding="max_length",
                       truncation=True, return_tensors="pt")
    with torch.no_grad():
        return model.encode_text(
            tokens["input_ids"].to(DEVICE),
            tokens["attention_mask"].to(DEVICE)
        )  # (1, 256)

def query(text, top_k=6, save_path=None):
    txt_emb = encode_text(text)
    sims = (img_emb.to(DEVICE) @ txt_emb.T).squeeze(1).cpu()
    top_idx = sims.argsort(descending=True)[:top_k].tolist()
    top_scores = sims[top_idx].tolist()

    fig, axes = plt.subplots(1, top_k, figsize=(3.2 * top_k, 3.8))
    fig.suptitle(f'🔍  "{text}"', fontsize=13, fontweight="bold", y=1.03)

    for rank, (ax, idx, score) in enumerate(zip(axes, top_idx, top_scores)):
        img = ds[idx]["image"].convert("RGB")
        ax.imshow(img)
        ax.set_title(f"#{rank+1}  {score:.3f}", fontsize=9,
                     color="#27ae60" if rank == 0 else "#555")
        cap = captions[idx]
        ax.set_xlabel(cap[:55] + "…" if len(cap) > 55 else cap,
                      fontsize=6.5, color="#444")
        ax.set_xticks([]); ax.set_yticks([])
        color = "#2ecc71" if rank == 0 else "#bbb"
        lw = 3 if rank == 0 else 1
        for s in ax.spines.values():
            s.set_edgecolor(color); s.set_linewidth(lw)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()

In [ ]:
query("a dog playing in the snow", save_path="query_dog_snow.png")

In [ ]:
query("people eating at a restaurant")

In [ ]:
query("a child riding a bicycle")

In [ ]:
query("a sunset over the ocean")

In [ ]:
query("a crowded city street at night")

---
## 4. Similarity matrix — positive vs negative pairs

Pick `n` random image-caption pairs. The matrix entry `[i, j]` is the cosine similarity between image `i` and caption `j`.  
**Diagonal = positive pairs** (should be bright). **Off-diagonal = negatives** (should be dark).  
Lime boxes highlight the ground-truth matches.

In [ ]:
def plot_similarity_matrix(n=16, seed=42, save_path=None):
    random.seed(seed)
    indices = random.sample(range(len(ds)), n)

    imgs, ids_list, masks = [], [], []
    row_captions = []
    for idx in indices:
        row = ds[idx]
        img = row["image"].convert("RGB")
        imgs.append(VAL_TRANSFORM(img))
        cap = row["caption"][0]
        row_captions.append(cap[:30] + "…" if len(cap) > 30 else cap)
        tok = tokenizer(cap, max_length=64, padding="max_length",
                        truncation=True, return_tensors="pt")
        ids_list.append(tok["input_ids"].squeeze(0))
        masks.append(tok["attention_mask"].squeeze(0))

    with torch.no_grad():
        ie = model.encode_image(torch.stack(imgs).to(DEVICE))
        te = model.encode_text(torch.stack(ids_list).to(DEVICE),
                               torch.stack(masks).to(DEVICE))
        # Raw cosine similarities (no temperature — easier to read)
        sim = (ie @ te.T).cpu().numpy()

    # Compute per-row stats for annotation
    diag = np.diag(sim)
    off_mask = ~np.eye(n, dtype=bool)
    off_mean = sim[off_mask].mean()
    off_std  = sim[off_mask].std()
    print(f"Positive (diagonal) sim:  mean={diag.mean():.3f}  std={diag.std():.3f}")
    print(f"Negative (off-diag)  sim:  mean={off_mean:.3f}  std={off_std:.3f}")
    print(f"Separation (pos−neg mean): {diag.mean() - off_mean:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(15, 6),
                             gridspec_kw={"width_ratios": [2, 1]})

    # ---- Left: full matrix -----------------------------------------------
    ax = axes[0]
    vabs = max(abs(sim.min()), abs(sim.max()))
    im = ax.imshow(sim, cmap="RdBu_r", vmin=-vabs, vmax=vabs, aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Cosine similarity")

    # Lime boxes on diagonal (positive pairs)
    for i in range(n):
        ax.add_patch(patches.Rectangle(
            (i - 0.5, i - 0.5), 1, 1,
            fill=False, edgecolor="#00ff88", linewidth=2.2
        ))

    ax.set_xticks(range(n))
    ax.set_xticklabels([f"T{i}" for i in range(n)], fontsize=7, rotation=45, ha="right")
    ax.set_yticks(range(n))
    ax.set_yticklabels([f"I{i}" for i in range(n)], fontsize=7)
    ax.set_xlabel("Text index", fontsize=11)
    ax.set_ylabel("Image index", fontsize=11)
    ax.set_title(f"Image–Text cosine similarity matrix  (n={n})\n"
                 f"Lime = positive (diagonal) pairs",
                 fontsize=12, fontweight="bold")

    # ---- Right: distribution of pos vs neg scores ------------------------
    ax2 = axes[1]
    neg_scores = sim[off_mask]
    pos_scores = diag
    bins = np.linspace(sim.min() - 0.02, sim.max() + 0.02, 40)
    ax2.hist(neg_scores, bins=bins, color="#e74c3c", alpha=0.7,
             label=f"Negatives (n={len(neg_scores)})", density=True)
    ax2.hist(pos_scores, bins=bins, color="#2ecc71", alpha=0.9,
             label=f"Positives (n={len(pos_scores)})", density=True)
    ax2.axvline(diag.mean(), color="#27ae60", lw=2, linestyle="--",
                label=f"pos mean={diag.mean():.3f}")
    ax2.axvline(off_mean, color="#c0392b", lw=2, linestyle="--",
                label=f"neg mean={off_mean:.3f}")
    ax2.set_xlabel("Cosine similarity", fontsize=11)
    ax2.set_ylabel("Density", fontsize=11)
    ax2.set_title("Score distribution:\nPositives vs Negatives",
                  fontsize=12, fontweight="bold")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()

plot_similarity_matrix(n=16, save_path="similarity_matrix.png")

In [ ]:
# Larger matrix for the blog — shows more structure
plot_similarity_matrix(n=32, seed=7, save_path="similarity_matrix_32.png")

---
## 5. Recall@k on the validation set

In [ ]:
# Use 1000 held-out examples (same split as training val)
random.seed(42)
all_idx = list(range(len(ds)))
random.shuffle(all_idx)
VAL_IDX = all_idx[:1000]

def compute_recalls(indices, batch_size=128):
    all_ie, all_te = [], []
    for start in tqdm(range(0, len(indices), batch_size), desc="Val embeddings"):
        batch_idx = indices[start:start + batch_size]
        imgs, ids_list, masks = [], [], []
        for idx in batch_idx:
            row = ds[idx]
            imgs.append(VAL_TRANSFORM(row["image"].convert("RGB")))
            tok = tokenizer(row["caption"][0], max_length=64, padding="max_length",
                            truncation=True, return_tensors="pt")
            ids_list.append(tok["input_ids"].squeeze(0))
            masks.append(tok["attention_mask"].squeeze(0))
        with torch.no_grad():
            all_ie.append(model.encode_image(torch.stack(imgs).to(DEVICE)))
            all_te.append(model.encode_text(torch.stack(ids_list).to(DEVICE),
                                            torch.stack(masks).to(DEVICE)))
    ie = torch.cat(all_ie)  # (N, 256)
    te = torch.cat(all_te)
    sims = ie @ te.T        # (N, N)
    targets = torch.arange(len(ie), device=ie.device)
    ranks = sims.argsort(dim=1, descending=True)
    results = {}
    for k in [1, 5, 10]:
        hits = (ranks[:, :k] == targets.unsqueeze(1)).any(dim=1)
        results[f"R@{k}"] = hits.float().mean().item()
    return results

recalls = compute_recalls(VAL_IDX)
for k, v in recalls.items():
    print(f"  {k}: {v:.3f}  ({v*100:.1f}%)")

---
## 6. Failure cases — where does it go wrong?

Find examples where the correct image is NOT the top-1 result.

In [ ]:
def show_failures(n_failures=4, save_path=None):
    """
    For each failure: show the query caption, the retrieved #1 image (wrong),
    and the true matching image side by side.
    """
    sample_idx = random.sample(VAL_IDX, 200)
    failures = []

    for idx in tqdm(sample_idx, desc="Finding failures"):
        row = ds[idx]
        cap = row["caption"][0]
        txt_emb = encode_text(cap)
        sims = (img_emb.to(DEVICE) @ txt_emb.T).squeeze(1).cpu()
        top1_idx = sims.argmax().item()
        if top1_idx != idx:  # wrong retrieval
            failures.append((idx, top1_idx, sims[top1_idx].item(), cap))
        if len(failures) >= n_failures:
            break

    print(f"Found {len(failures)} failures from 200 queries")
    fig, axes = plt.subplots(n_failures, 2, figsize=(7, 3.5 * n_failures))
    fig.suptitle("Failure cases — retrieved vs ground-truth",
                 fontsize=13, fontweight="bold")

    for row_i, (true_idx, pred_idx, score, cap) in enumerate(failures):
        for col_i, (img_idx, title, color) in enumerate([
            (pred_idx, f"Retrieved  (sim={score:.3f})", "#e74c3c"),
            (true_idx, "Ground truth", "#2ecc71"),
        ]):
            ax = axes[row_i][col_i]
            ax.imshow(ds[img_idx]["image"].convert("RGB"))
            ax.set_title(title, fontsize=9, color=color, fontweight="bold")
            if col_i == 0:
                ax.set_ylabel(f'"{cap[:40]}…"' if len(cap) > 40 else f'"{cap}"',
                              fontsize=7.5, color="#333", labelpad=6)
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_edgecolor(color); s.set_linewidth(2)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()

show_failures(n_failures=4, save_path="failure_cases.png")

---
## 7. Interactive widget

In [ ]:
import ipywidgets as widgets
from IPython.display import display

text_box = widgets.Text(value="a horse jumping over a fence",
                        description="Query:",
                        layout=widgets.Layout(width="55%"))
k_slider = widgets.IntSlider(value=6, min=1, max=12, description="Top-k:")
btn = widgets.Button(description="Search", button_style="primary")
out = widgets.Output()

def on_search(b):
    with out:
        out.clear_output(wait=True)
        query(text_box.value, top_k=k_slider.value)

btn.on_click(on_search)
display(widgets.VBox([widgets.HBox([text_box, k_slider]), btn, out]))